In [1]:
!pip uninstall -y transformers
!pip install "transformers==4.50.0"

Found existing installation: transformers 5.16.1
Uninstalling transformers-5.16.1:
  Successfully uninstalled transformers-5.16.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 whic

In [2]:
import transformers
print(transformers.__version__)

4.50.0


DistilBERT

In [3]:
import pandas as pd
import numpy as np
import tf_keras as keras
import tensorflow as tf
from transformers import RobertaTokenizer,TFRobertaForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report
from sklearn.preprocessing import OneHotEncoder

In [4]:
df = pd.read_csv(r"/content/sample_data/IMDB Dataset.csv")

print(df.head(5))

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [5]:
oe = OneHotEncoder(drop="first",sparse_output=False)
y = oe.fit_transform(df[["sentiment"]])

X = df["review"]

X_train_val,X_test,y_train_val,y_test = train_test_split(
    X.tolist(),y.tolist(),train_size=5000,test_size=1000,random_state=42,stratify=y
)

X_train,X_val,y_train,y_val = train_test_split(
    X_train_val,y_train_val,test_size=500,random_state=42,stratify=y_train_val
)

In [7]:
Tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [8]:
train_encodings = dict(Tokenizer(X_train,padding=True,truncation=True,max_length=128,return_tensors="tf"))
val_encodings = dict(Tokenizer(X_val,padding=True,truncation=True,max_length=128,return_tensors="tf"))
test_encodings = dict(Tokenizer(X_test,padding=True,truncation=True,max_length=128,return_tensors="tf"))
train_label = tf.convert_to_tensor(y_train)
val_label = tf.convert_to_tensor(y_val)
test_label = tf.convert_to_tensor(y_test)

In [9]:
Model = TFRobertaForSequenceClassification.from_pretrained("roberta-base",num_labels=2)
optimizer = keras.optimizers.Adam(learning_rate=2e-5)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaForSequenceClassification: ['roberta.embeddings.position_ids']
- This IS expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFRobertaForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predicti

In [10]:
Model.compile(optimizer=optimizer,loss=loss,metrics=["accuracy"])
history = Model.fit(train_encodings,train_label,validation_data=(val_encodings,val_label),epochs=10)

Epoch 1/10
141/141 [==============================] - 176s 884ms/step - loss: 0.4305 - accuracy: 0.7833 - val_loss: 0.2589 - val_accuracy: 0.9000
Epoch 2/10
141/141 [==============================] - 123s 872ms/step - loss: 0.2358 - accuracy: 0.9144 - val_loss: 0.3265 - val_accuracy: 0.8760
Epoch 3/10
141/141 [==============================] - 123s 872ms/step - loss: 0.1672 - accuracy: 0.9404 - val_loss: 0.2843 - val_accuracy: 0.8920
Epoch 4/10
141/141 [==============================] - 123s 872ms/step - loss: 0.1052 - accuracy: 0.9680 - val_loss: 0.2864 - val_accuracy: 0.9040
Epoch 5/10
141/141 [==============================] - 123s 873ms/step - loss: 0.0834 - accuracy: 0.9742 - val_loss: 0.4941 - val_accuracy: 0.8580
Epoch 6/10
141/141 [==============================] - 123s 872ms/step - loss: 0.0554 - accuracy: 0.9824 - val_loss: 0.3796 - val_accuracy: 0.8880
Epoch 7/10
141/141 [==============================] - 123s 874ms/step - loss: 0.0388 - accuracy: 0.9891 - val_loss: 0.3810 -

In [11]:
pred = Model.predict(test_encodings)
pred = pred.logits
probabilities = tf.nn.softmax(pred,axis=1)
predictions = tf.argmax(probabilities,axis=1)

32/32 [==============================] - 12s 283ms/step


In [12]:
accuracy = accuracy_score(predictions,test_label)
print("Accuracy score: ",accuracy)
report = classification_report(predictions,test_label)
print("Classification report: ")
print(report)

Accuracy score:  0.91
Classification report: 
              precision    recall  f1-score   support

           0       0.88      0.93      0.91       474
           1       0.94      0.89      0.91       526

    accuracy                           0.91      1000
   macro avg       0.91      0.91      0.91      1000
weighted avg       0.91      0.91      0.91      1000

